# Smart Waste Classification System

# Notebook 02 — Dataset Validation

**Author:** Syed Soban Mudassar

---

## Purpose

This notebook validates the TrashNet dataset before exploratory data analysis
and model training.

The validation process checks whether the dataset is complete, readable, and
organized correctly.

This stage is necessary because machine learning models depend on the quality
of the data they receive.

In [2]:
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent
dataset_path = project_root / "data" / "raw" / "trashnet"

print("Current working directory:", current_dir)
print("Project root:", project_root)
print("Dataset path:", dataset_path)
print("Dataset exists:", dataset_path.exists())

Current working directory: c:\Users\Mudassar Iqbal Shah\Documents\Internship\smart-waste-classification\notebooks
Project root: c:\Users\Mudassar Iqbal Shah\Documents\Internship\smart-waste-classification
Dataset path: c:\Users\Mudassar Iqbal Shah\Documents\Internship\smart-waste-classification\data\raw\trashnet
Dataset exists: True


In [3]:
if dataset_path.exists():
    print("Dataset is ready for validation.")
else:
    print("Dataset folder was not found. Check the path before continuing.")

Dataset is ready for validation.


##  Image Integrity, Dimensions, and Formats

After confirming the folder structure and image counts, every image must be
opened and checked.

This stage will:

- detect corrupted image files
- record image dimensions
- record image formats
- continue processing even if an invalid image is found

Exception handling is used so that one damaged image does not stop the entire
validation process.

In [6]:
EXPECTED_CLASSES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

print("Expected classes:")
print(EXPECTED_CLASSES)

print("\nSupported image formats:")
print(IMAGE_EXTENSIONS)

Expected classes:
['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

Supported image formats:
{'.jpeg', '.jpg', '.png'}


In [4]:
from collections import Counter

from PIL import Image, UnidentifiedImageError

In [7]:
corrupted_images = []
image_dimensions = Counter()
image_formats = Counter()
validated_images = 0

for class_name in EXPECTED_CLASSES:
    class_path = dataset_path / class_name

    image_files = [
        file_path
        for file_path in class_path.iterdir()
        if file_path.is_file()
        and file_path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for image_path in image_files:
        try:
            # First opening checks whether the internal image data is valid.
            with Image.open(image_path) as image:
                image.verify()

            # Reopen the image after verify() to read its metadata safely.
            with Image.open(image_path) as image:
                width, height = image.size
                image_format = image.format or "Unknown"

            image_dimensions[(width, height)] += 1
            image_formats[image_format] += 1
            validated_images += 1

        except (UnidentifiedImageError, OSError, ValueError) as error:
            corrupted_images.append(
                {
                    "path": str(image_path),
                    "error": str(error),
                }
            )

print("Validation completed.")
print("Successfully validated images:", validated_images)
print("Corrupted images:", len(corrupted_images))

Validation completed.
Successfully validated images: 2527
Corrupted images: 0


In [8]:
print("Most common image dimensions:")
print("-" * 40)

for dimensions, count in image_dimensions.most_common(10):
    width, height = dimensions
    print(f"{width} × {height}: {count} images")

print("\nImage formats:")
print("-" * 40)

for image_format, count in image_formats.items():
    print(f"{image_format}: {count} images")

Most common image dimensions:
----------------------------------------
512 × 384: 2527 images

Image formats:
----------------------------------------
JPEG: 2527 images


##  Image Validation Observation

All dataset images were checked using Pillow.

The validation process confirmed:

- all images could be opened successfully
- no corrupted images were detected
- the images use a consistent JPEG format
- the images have consistent or highly similar dimensions

Consistent dimensions simplify the preprocessing process, although the images
will still be resized to the input size required by the selected neural
network architecture.

The absence of corrupted images means that the dataset can safely proceed to
duplicate detection and exploratory data analysis.

##  Duplicate Image Detection

Duplicate images can cause data leakage and misleading evaluation results.

For example, if the same image appears in both the training set and the test
set, the model may appear to perform well because it has already seen that
image during training.

Exact duplicates can be detected by calculating a cryptographic hash for each
file.

Files with the same hash contain the same binary content.

In [9]:
import hashlib
from collections import defaultdict

hash_to_paths = defaultdict(list)

for class_name in EXPECTED_CLASSES:
    class_path = dataset_path / class_name

    image_files = [
        file_path
        for file_path in class_path.iterdir()
        if file_path.is_file()
        and file_path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for image_path in image_files:
        sha256 = hashlib.sha256()

        with image_path.open("rb") as image_file:
            while chunk := image_file.read(8192):
                sha256.update(chunk)

        file_hash = sha256.hexdigest()
        hash_to_paths[file_hash].append(image_path)

duplicate_groups = {
    file_hash: paths
    for file_hash, paths in hash_to_paths.items()
    if len(paths) > 1
}

duplicate_file_count = sum(
    len(paths) - 1
    for paths in duplicate_groups.values()
)

print("Duplicate groups:", len(duplicate_groups))
print("Extra duplicate files:", duplicate_file_count)

Duplicate groups: 3
Extra duplicate files: 3


In [10]:
if duplicate_groups:
    print("Exact duplicate groups:\n")

    for group_number, paths in enumerate(
        duplicate_groups.values(),
        start=1,
    ):
        print(f"Group {group_number}")

        for path in paths:
            print(" -", path.relative_to(project_root))

        print()
else:
    print("No exact duplicate images were found.")

Exact duplicate groups:

Group 1
 - data\raw\trashnet\glass\glass115.jpg
 - data\raw\trashnet\metal\metal91.jpg

Group 2
 - data\raw\trashnet\glass\glass176.jpg
 - data\raw\trashnet\plastic\plastic152.jpg

Group 3
 - data\raw\trashnet\glass\glass389.jpg
 - data\raw\trashnet\plastic\plastic332.jpg



## Observation

The validation process detected three groups of exact duplicate images.

The duplicates occur across different class folders, suggesting that identical
image files exist under different labels.

The raw dataset will remain unchanged.

The duplicate images have been documented and will be considered during the
preprocessing stage.